# AI-Based Delivery Delay Risk Dataset Generation
This notebook creates a synthetic dataset for modeling multi-class delivery delay risk in e-commerce logistics.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(96)

n = 200_000

In [ ]:
# Core structural features
regions = ['North', 'South', 'East', 'West', 'Central']
delivery_types = ['Standard', 'Express', 'Same-Day']

data = pd.DataFrame({
    "day_of_week": np.random.randint(0, 7, n),
    "is_peak_hour": np.random.randint(0, 2, n),
    "warehouse_region": np.random.choice(regions, n),
    "delivery_type": np.random.choice(delivery_types, n)
})

In [ ]:
def generate_distance(region):
    if region == 'North':
        return np.random.uniform(800, 3000)
    elif region == 'East':
        return np.random.uniform(5, 1200)
    elif region == 'Central':
        return np.random.uniform(200, 2000)
    else:
        return np.random.uniform(100, 2500)

data["shipment_distance_km"] = data["warehouse_region"].apply(generate_distance)

In [ ]:
def generate_traffic(is_peak):
    if is_peak == 1:
        return np.random.choice([3,4,5], p=[0.3,0.4,0.3])
    else:
        return np.random.choice([1,2,3], p=[0.4,0.4,0.2])

data["traffic_level"] = data["is_peak_hour"].apply(generate_traffic)

In [ ]:
data["weather_severity"] = np.random.choice(
    [0,1,2,3],
    size=n,
    p=[0.5,0.25,0.15,0.10]
)

In [ ]:
def generate_courier_load(delivery_type):
    if delivery_type == "Same-Day":
        return np.random.uniform(70, 100)
    elif delivery_type == "Express":
        return np.random.uniform(50, 90)
    else:
        return np.random.uniform(30, 80)

data["courier_load_pct"] = data["delivery_type"].apply(generate_courier_load)

In [ ]:
def generate_warehouse_time(load, delivery_type):
    base = np.random.uniform(2, 24)
    if delivery_type == "Same-Day":
        base += 5
    return base + (load / 100) * 5

data["warehouse_time_hrs"] = data.apply(
    lambda row: generate_warehouse_time(row["courier_load_pct"], row["delivery_type"]),
    axis=1
)

In [ ]:
def generate_order_volume(region):
    if region == "Central":
        return np.random.randint(300, 600)
    elif region == "North":
        return np.random.randint(200, 500)
    else:
        return np.random.randint(100, 400)

data["order_volume"] = data["warehouse_region"].apply(generate_order_volume)

In [ ]:
def generate_past_delay(weather, region):
    base = np.random.uniform(0.05, 0.3)
    if weather >= 2:
        base += 0.1
    if region == "North":
        base += 0.05
    return min(base, 0.6)

data["past_delay_rate"] = data.apply(
    lambda row: generate_past_delay(row["weather_severity"], row["warehouse_region"]),
    axis=1
)

In [ ]:
data["order_volume_norm"] = data["order_volume"] / data["order_volume"].max()
data["warehouse_time_norm"] = data["warehouse_time_hrs"] / data["warehouse_time_hrs"].max()
data["distance_norm"] = data["shipment_distance_km"] / data["shipment_distance_km"].max()
data["traffic_norm"] = data["traffic_level"] / 5
data["weather_norm"] = data["weather_severity"] / 3
data["courier_load_norm"] = data["courier_load_pct"] / 100
data["past_delay_norm"] = data["past_delay_rate"] / 0.6

In [ ]:
data["risk_score"] = (
    0.10 * data["order_volume_norm"] +
    0.12 * data["warehouse_time_norm"] +
    0.12 * data["distance_norm"] +
    0.18 * data["traffic_norm"] +
    0.12 * data["weather_norm"] +
    0.10 * data["courier_load_norm"] +
    0.16 * data["past_delay_norm"]
)

In [ ]:
# Severe traffic + bad weather
mask1 = (data["traffic_level"] >= 4) & (data["weather_severity"] >= 2)
# data.loc[mask1, "risk_score"] += 0.15
data.loc[mask1, "risk_score"] += 0.18

# Same-day under high load
mask2 = (data["delivery_type"] == "Same-Day") & (data["courier_load_pct"] > 85)
# data.loc[mask2, "risk_score"] += 0.10
data.loc[mask2, "risk_score"] += 0.12

# High historical delay
mask3 = data["past_delay_rate"] > 0.5
# data.loc[mask3, "risk_score"] += 0.12
data.loc[mask3, "risk_score"] += 0.15

In [ ]:
# data["risk_score"] *= 0.85

# noise = np.random.normal(0, 0.04, n)
noise = np.random.normal(0, 0.025, n)
data["risk_score"] += noise

data["risk_score"] = data["risk_score"].clip(0, 1)

In [ ]:
def assign_status(risk):
    if risk < 0.50:
        return "On-Time"
    elif risk < 0.70:
        return "At Risk"
    else:
        return "Delayed"

data["delivery_status"] = data["risk_score"].apply(assign_status)

In [ ]:
data.drop(columns=["risk_score"], inplace=True)

In [ ]:
data.drop(columns=[
    "order_volume_norm",
    "warehouse_time_norm",
    "distance_norm",
    "traffic_norm",
    "weather_norm",
    "courier_load_norm",
    "past_delay_norm"
], inplace=True)

In [ ]:
print(data["delivery_status"].value_counts())
print(data.shape)
data.head()

delivery_status
On-Time    136182
At Risk     48437
Delayed     15381
Name: count, dtype: int64
(200000, 12)


,day_of_week,is_peak_hour,warehouse_region,delivery_type,shipment_distance_km,traffic_level,weather_severity,courier_load_pct,warehouse_time_hrs,order_volume,past_delay_rate,delivery_status
0,6,1,South,Standard,2300.325466,4,2,39.181324,12.362810,182,0.254594,At Risk
1,3,0,West,Express,2434.495361,1,0,60.345142,19.393527,356,0.193293,On-Time
2,4,0,North,Same-Day,1853.785723,1,0,92.917023,15.851617,407,0.248329,On-Time
3,6,1,North,Same-Day,1433.376964,4,3,88.460255,31.295919,333,0.420760,Delayed
4,2,0,South,Express,691.077648,2,3,82.512752,25.896243,133,0.258417,At Risk


In [ ]:
# ------------------------------------------------------------
# SAVE DATASET TO CSV
# ------------------------------------------------------------
data.to_csv("raw.csv", index=False)

In [ ]:
from google.colab import files
files.download("raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>